In [1]:
import sys
import os

In [2]:
# Add the target folder to Python's search path
sys.path.insert(0, os.path.abspath('../trmnl-agent/app'))

from utils import day_of_week, get_headers, hex_to_grayscale

import datetime
import requests
from bs4 import BeautifulSoup

In [12]:
SOURCE_URL = os.getenv("SOURCE_URL", "https://www.accuweather.com/en/us/lakeville/55044/sinus-weather/2247734")

def get_page(target_url, pull_retry=1):
    """
    Extract content from the target URL.
    """
    retry_limit = max(1, int(pull_retry))
    retry_count = 0

    while retry_count < retry_limit:
        try:
            headers = get_headers()
            r = requests.get(target_url, headers=headers, verify=False, timeout=30)
            r.raise_for_status()
            return r.content
        except requests.RequestException as e:
            retry_count += 1
    return None


def parse_element(el):
    date = datetime.datetime.strptime(el["data-xvalue"], "%m/%d/%Y %H:%M:%S")
    return {"y": el["data-yvalue"],
            "color": hex_to_grayscale(el["fill"]),
            "day": day_of_week(date),
            "date": str(date.date())}


def extract_content(page_content):
    """
    Extract the chart bar data from the source page.
    """
    if page_content is None:
        raise ValueError("No page content was returned from the source URL.")

    soup = BeautifulSoup(page_content, features="html.parser")
    if soup.body is None:
        raise ValueError("Source page did not contain a body element.")

    chart_bars = soup.body.find_all('rect', class_='index-chart-bar')
    if not chart_bars:
        raise ValueError("Source page did not contain any index-chart-bar elements.")

    return [parse_element(bar) for bar in chart_bars]


def build_trmnl_payload(extracted_data):
    core_payload = {x:extracted_data[x] for x in range(len(extracted_data))}
    return {"merge_variables": core_payload}


def webhook_upload(target_url, data):
    """
    Send extracted data to the configured webhook URL.
    """
    try:
        response = requests.post(target_url, json=data, verify=False)
        response.raise_for_status()
        #logging.debug(f"Webhook response from {target_url}: {response.status_code} - {response.text}")
        #logging.info("Data successfully sent to webhook.")
    except requests.RequestException as e:
        #logging.error(f"Error sending data to webhook: {e}")
        print(e)


# def run(cfg):
#     #logging.config.dictConfig(cfg.LOG_CONFIG)
#     #logger = logging.getLogger(__name__)
#     #logger.info("TRMNL Agent starting up")

#     pull_retry = getattr(cfg, "PULL_RETRY", None)
#     if isinstance(pull_retry, (int, str)):
#         page_content = get_page(cfg.SOURCE_URL, pull_retry)
#     else:
#         page_content = get_page(cfg.SOURCE_URL)
#     if page_content is None:
#         message = f"Failed to fetch source data from {cfg.SOURCE_URL}"
#         logger.error(message)
#         raise RuntimeError(message)

#     try:
#         extracted_data = extract_content(page_content)
#     except (AttributeError, KeyError, TypeError, ValueError) as exc:
#         logger.exception("Source data extraction failed for %s", cfg.SOURCE_URL)
#         raise RuntimeError(f"Failed to extract source data from {cfg.SOURCE_URL}") from exc

#     #logger.info("Source Data Extracted Successfully")
#     data = build_trmnl_payload(extracted_data)
#     logger.debug(f"Extracted data: {data}")
#     webhook_upload(cfg.WEBHOOK_URL, data)
#     logger.info("Upload Completed. TRMNL Agent finished execution")



In [4]:
page_content = get_page(SOURCE_URL, 3)

/opt/homebrew/lib/python3.11/site-packages/urllib3/connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.accuweather.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [14]:
extracted_data = extract_content(page_content)

In [15]:
extracted_data

[{'y': '3', 'color': '#7C7C7C', 'day': 'Thr', 'date': '2026-06-18'},
 {'y': '2', 'color': '#CBCBCB', 'day': 'Fri', 'date': '2026-06-19'},
 {'y': '1', 'color': '#A1A1A1', 'day': 'Sat', 'date': '2026-06-20'},
 {'y': '1', 'color': '#A1A1A1', 'day': 'Sun', 'date': '2026-06-21'},
 {'y': '2', 'color': '#CBCBCB', 'day': 'Mon', 'date': '2026-06-22'},
 {'y': '1', 'color': '#A1A1A1', 'day': 'Tue', 'date': '2026-06-23'},
 {'y': '1', 'color': '#A1A1A1', 'day': 'Wed', 'date': '2026-06-24'}]

In [11]:
str(datetime.datetime.now().date())

'2026-06-18'